# xai

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/06_detection-segmentation-visualization'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
![ -f utils.py ] || gdown --id 1NUGtNIABwt4PSAkBXPcMtnKPPaZYmxP3 -O utils.py
![ -f chart.py ] || gdown --id 1Y8zhWoaLEkeiAIkpEQJL_FsQInc7i-Os -O chart.py

In [ ]:
%pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.4.8-py3-none-any.whl size=38244 sha256=949a84752c3c524003f90cfd6eb7f1e0bd125121cc612ebd052aa6401accb6a8
  Stored in directory: /root/.cache/pip/wheels/f8/04/36/94ff3c8a4215826a21946b34c01180817e606989fdf53f7cd6
Successfully built grad-cam


In [ ]:
from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.io import read_image
from torchvision import transforms

import torch
import numpy as np

from chart import show

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Resnet model
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device)
preprocess = ResNet50_Weights.IMAGENET1K_V2.transforms() # applied

# Target layer: last conv layer
target_layers = [model.layer4[-1]]

# Input image - https://github.com/EliSchwartz/imagenet-sample-images
# image = read_image("n01443537_goldfish.jpeg") # as a tensor [0, 255]
image = read_image("shitzu.jpeg") # as a tensor [0, 255]
# image = read_image("n02701002_ambulance.jpeg")
# image = read_image("dog_cat.jfif")
show(image.numpy().transpose([1, 2, 0]), scale=0.3)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-3-1f7e872b382e>", line 1, in <cell line: 1>
    from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
  File "<frozen importlib._bootstrap>", line 1027, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1002, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 945, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1439, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1408, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1366, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/l

Adapted from [Advanced AI explainability for PyTorch
](https://github.com/jacobgil/pytorch-grad-cam)

In [ ]:
# Create an input tensor image for your model
batch = preprocess(image).unsqueeze(0).to(device) # channels first
# Note: input_tensor can be a batch tensor with several images!

# Construct the CAM object once, and then re-use it on many images:
cam = GradCAM(model=model, target_layers=target_layers)

# You can also use it within a with statement, to make sure it is freed,
# In case you need to re-create it inside an outer loop:
# with GradCAM(model=model, target_layers=target_layers) as cam:
#   ...

# We have to specify the target we want to generate
# the Class Activation Maps for.
# If targets is None, the highest scoring category
# will be used for every image in the batch.
# Here we use ClassifierOutputTarget, but you can define your own custom targets
# That are, for example, combinations of categories, or specific outputs in a non standard model.

# class id /labels
# 1 goldfish
# 155 Shih-Tzu
# 207 golden retriever
# 407 ambulance
targets = None
# Uncomment the following line with you want to visualize the map for a particular class
# targets = [ClassifierOutputTarget(1)] # https://deeplearning.cms.waikato.ac.nz/user-guide/class-maps/IMAGENET/

# You can also pass aug_smooth=True and eigen_smooth=True, to apply smoothing.
grayscale_cam = cam(input_tensor=batch, targets=targets)
# In this example grayscale_cam has only one image in the batch:
grayscale_cam = grayscale_cam[0, :]

# Prediction
prediction = model(batch).squeeze(0).softmax(0)
class_id = prediction.argmax().item()
print(f"Class id={class_id}")

center_crop = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
])

normalized_image = center_crop(image).numpy().transpose([1, 2, 0]) / 255.
# print(normalized_image)
visualization = show_cam_on_image(normalized_image, grayscale_cam, use_rgb=True)
show(visualization, scale=0.3)


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-4-7d136a987436>", line 2, in <cell line: 2>
    batch = preprocess(image).unsqueeze(0).to("cuda") # channels first
NameError: name 'preprocess' is not defined

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
AttributeError: 'NameError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_getinnerframes(etb, number_of_lines_of_context, tb_offset)
  File "/usr/lo